## Load the summary information and preprocessed data

In [1]:
import os
import json
import pandas as pd

from pathlib import Path

from single_sample_gsea import ss_gsea


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 32 samples and 270 columns.
First few rows of the data:


,living_day,die,Etiology,Degree_of_tumor_differentiation,AFP(ng/mL),TNM_stages,BCLC,Tumor_size(mm),age,Satellites,...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,176,1,Cirrhosis,Poor,30783,Ⅳ,B,105,37,Present,...,4.536021,2.249830,3.781962,5.140040,3.470682,3.293235,4.711250,4.434599,2.847444,4.243018
1,190,1,Hepatitis_B,Poor,2668,Ⅳ,C,190,44,Present,...,3.924160,1.357657,2.073493,5.633265,2.004485,2.394021,3.718035,3.279829,1.996515,3.182071
2,4,1,Hepatitis_B,Poor,>121000,Ⅲ,C,230,43,Absent,...,6.147698,3.642578,6.042068,6.485741,5.608361,5.945016,7.058042,6.250611,4.534191,5.931232
3,519,1,Hepatitis_B,Poor,1.01,Ⅳ,C,130,64,Absent,...,3.920815,1.615761,3.311212,4.523287,2.619709,2.821889,4.293913,3.627103,2.215886,3.478311
4,533,1,Hepatitis_B,Poor,3710,Ⅳ,C,120,48,Absent,...,4.526706,2.097380,4.076484,5.227762,3.436344,3.337557,5.054281,4.396321,2.804797,4.459888


## Load gene sets

In [2]:
MSIGDB_JSON = Path("../../resources/h.all.v2026.1.Hs.json")

with MSIGDB_JSON.open(encoding="utf-8") as f:
    gene_sets = {name: data["geneSymbols"] for name, data in json.load(f).items()}

print(f"Loaded {len(gene_sets)} gene sets.")


Loaded 50 gene sets.


## Compute single sample GSEA on the preprocessed data

Select only gene columns for the computation.

In [3]:
# IMPORTANT: This is dataset-specific and is the only variable that needs to be changed for each of the dataset.
sample_id_column = 'sample_title'

gene_expression_df = df.iloc[:, summary_data['data_summary']['clinical_features']:].copy()
gene_expression_df.index = df[sample_id_column].astype(str)
gene_expression_df = gene_expression_df.T
gene_expression_df.index.name = "gene"

result = ss_gsea(gene_expression_df, gene_sets)


/Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/.venv/lib/python3.14/site-packages/single_sample_gsea/ssgsea.py:51: RuntimeWarning: invalid value encountered in divide
  signature_ecdf = signature_ecdf / np.sum(signature_ecdf, axis=0)


## Merge the results with the clinical data

The output table should have the same number of rows as before and columns should sum up to clinical data + 50 gene sets.

In [4]:
clinical_df = df.iloc[:, :summary_data["data_summary"]["clinical_features"]].copy()
clinical_df = clinical_df.set_index(sample_id_column)

output_df = pd.concat([clinical_df, result], axis=1)
output_df = output_df.reset_index(drop=False)

output_df.to_csv(f"{GEO_ID}_preprocessed_ssgsea.csv", index=False)

# sanity check (there is 50 gene sets)
assert output_df.shape[1] == summary_data["data_summary"]["clinical_features"] + 50
